In [ ]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

# 1. CONEXIÓN Y CARGA
drive.mount('/content/drive')
path = '/content/drive/MyDrive/Brazilian E-Commerce'

def cargar_datos(ruta):
    archivos = [f for f in os.listdir(ruta) if f.endswith('.csv')]
    datasets = {}
    for f in archivos:
        name = f.replace('.csv', '')
        datasets[name] = pd.read_csv(os.path.join(ruta, f))
    return datasets

data = cargar_datos(path)

# 2. UNIFICACIÓN (MERGE) - Construcción de la tabla
df = data['olist_orders_dataset'].merge(data['olist_order_items_dataset'], on='order_id', how='left')
df = df.merge(data['olist_customers_dataset'], on='customer_id', how='left')
df = df.merge(data['olist_products_dataset'], on='product_id', how='left')
df = df.merge(data['product_category_name_translation'], on='product_category_name', how='left')
df = df.merge(data['olist_order_reviews_dataset'], on='order_id', how='left')

# 3. TRATAMIENTO DE FECHAS
cols_fecha = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in cols_fecha:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# 4. LÓGICA DE NEGOCIO Y LIMPIEZA DE NULOS
# Calculamos días. Si no hay fecha (cancelados/proceso), el resultado es NaN temporalmente.
df['dias_entrega_num'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df['retraso_logistico_num'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days

# IMPUTACIÓN: Convertimos NaN a 0
df['dias_entrega_num'] = df['dias_entrega_num'].fillna(0)
df['retraso_logistico_num'] = df['retraso_logistico_num'].fillna(0)
df['price'] = df['price'].fillna(0)
df['freight_value'] = df['freight_value'].fillna(0)
df['product_category_name_english'] = df['product_category_name_english'].fillna('Sin Información')

# 5. CREACIÓN DE ETIQUETAS DESCRIPTIVAS
def categorizar_estado(row):
    if row['order_status'] == 'delivered':
        return "Entrega Completada"
    elif row['order_status'] == 'canceled':
        return "No aplica: Pedido Cancelado"
    elif row['order_status'] == 'unavailable':
        return "No aplica: Stock No Disponible"
    elif row['order_status'] in ['invoiced', 'processing', 'approved']:
        return f"Pendiente: {row['order_status'].capitalize()}"
    elif row['order_status'] == 'shipped':
        return "En Tránsito: Enviado"
    else:
        return "Otros"

df['detalle_estado_logistico'] = df.apply(categorizar_estado, axis=1)

# 6. RENOMBRADO
traduccion = {
    'order_id': 'id_pedido',
    'customer_id': 'id_cliente_pedido',
    'customer_unique_id': 'id_unico_cliente',
    'order_status': 'estado_pedido',
    'order_purchase_timestamp': 'fecha_compra',
    'price': 'precio_producto',
    'freight_value': 'costo_envio',
    'product_category_name_english': 'categoria_producto',
    'customer_city': 'ciudad_cliente',
    'customer_state': 'estado_cliente',
    'review_score': 'puntaje_satisfaccion',
    'dias_entrega_num': 'dias_entrega',
    'retraso_logistico_num': 'retraso_dias',
    'detalle_estado_logistico': 'explicacion_nulos'
}
df_final = df.rename(columns=traduccion)

# Seleccionamos columnas finales para un archivo ligero
columnas_finales = [
    'id_pedido',
    'id_unico_cliente',
    'estado_pedido',
    'fecha_compra',
    'precio_producto',
    'costo_envio',
    'categoria_producto',
    'ciudad_cliente',
    'estado_cliente',
    'puntaje_satisfaccion',
    'dias_entrega',
    'retraso_dias',
    'explicacion_nulos'
]
# 7. EXPORTACIÓN
df_final[columnas_finales].to_csv(os.path.join(path, 'reporte_maestro_olist_FINAL.csv'), index=False)

print(" ¡Proceso finalizado! El archivo 'reporte_maestro_olist_FINAL.csv' para Power BI.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ ¡Proceso finalizado! El archivo 'reporte_maestro_olist_FINAL.csv' está listo para Power BI.
